In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets torch scikit-learn accelerate

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_dataset_path() -> Path:
    drive_root = Path('/content/drive/MyDrive')
    candidates = list(drive_root.rglob('final_dataset.csv'))
    if not candidates:
        raise FileNotFoundError('final_dataset.csv was not found under /content/drive/MyDrive')
    return candidates[0]

def detect_columns(frame: pd.DataFrame) -> tuple[str, str]:
    code_candidates = ['code', 'snippet', 'text', 'source_code', 'content']
    label_candidates = ['label', 'type', 'authorship', 'class', 'target']
    code_col = next((column for column in code_candidates if column in frame.columns), None)
    label_col = next((column for column in label_candidates if column in frame.columns), None)
    if code_col is None or label_col is None:
        raise ValueError(f'Could not infer code and label columns from: {list(frame.columns)}')
    return code_col, label_col

def normalize_label(value) -> int:
    text = str(value).strip().lower()
    human_values = {'0', 'human', 'human-written', 'human_written', 'written', 'manual'}
    ai_values = {'1', 'ai', 'ai-generated', 'ai_generated', 'generated', 'machine', 'machine-generated', 'machine_generated'}
    if text in human_values:
        return 0
    if text in ai_values:
        return 1
    try:
        return int(float(text))
    except ValueError as error:
        raise ValueError(f'Unsupported label value: {value!r}') from error

DATA_PATH = find_dataset_path()
frame = pd.read_csv(DATA_PATH)
code_column, label_column = detect_columns(frame)
frame = frame[[code_column, label_column]].dropna().rename(columns={code_column: 'code', label_column: 'label'})
frame['code'] = frame['code'].astype(str)
frame['label'] = frame['label'].apply(normalize_label).astype(int)
train_df, test_df = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=frame['label'])
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.head()

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
MAX_LENGTH = 512

def tokenize_frame(data_frame: pd.DataFrame) -> Dataset:
    dataset = Dataset.from_pandas(data_frame[['code', 'label']].reset_index(drop=True))
    tokenized = dataset.map(
        lambda batch: tokenizer(batch['code'], truncation=True, padding='max_length', max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['code'],
    )
    tokenized = tokenized.rename_column('label', 'labels')
    tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized

train_dataset = tokenize_frame(train_df)
val_dataset = tokenize_frame(val_df)
test_dataset = tokenize_frame(test_df)

train_dataset[0] if len(train_dataset) else None

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)
model

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer, TrainingArguments

training_arguments = TrainingArguments(
    output_dir='/content/codebert_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
    seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', zero_division=0)
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(test_dataset)
test_logits = predictions.predictions
test_labels = predictions.label_ids
test_predictions = np.argmax(test_logits, axis=-1)

print('Confusion Matrix:')
print(confusion_matrix(test_labels, test_predictions))
print('
Classification Report:')
print(classification_report(test_labels, test_predictions, target_names=['Human-Written', 'AI-Generated']))

In [ ]:
output_dir = Path('/content/drive/MyDrive/models/codebert_finetuned')
output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
print(f'Saved CodeBERT model to {output_dir}')